# Phase 2B-3 · Animal health, mortality and carcass
Highwood Feeders simulated data. The placement population contains all modeled closeout outcomes. Carcass comparisons use finished sales. Treatment events join to calendar-dated scale intervals, while all financial summaries use unique animal IDs and original booked costs. All amounts are nominal CAD.

[Findings](../phase2b/03_HEALTH_CARCASS_FINDINGS.md) · [KPI definitions](../analysis_audit/04_KPI_DEFINITIONS.md) · [Baseline](../PHASE2A_CHECKPOINT.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from feedlot_analytics.carcass import carcass_bands, lot_health_review, sold_carcasses
from feedlot_analytics.closeout_economics import cost_structure
from feedlot_analytics.contracts import FINISHED, GRADES, MORTALITY
from feedlot_analytics.feed_growth import stage_coverage
from feedlot_analytics.health_growth import (
    first_pull_vs_stage, growth_landmark, intervals_with_treatments, persistent_growth, treatment_growth_pairs,
)
from feedlot_analytics.health_mortality import (
    DOF_BINS, DOF_LABELS, animal_history, daily_risk, load_health_analysis, protocol_review,
    risk_contrasts, risk_segments, treatment_followup,
)
from feedlot_analytics.health_statistics import cohort_difference, pen_rate_interval, within_lot_slope, with_entry_band
from feedlot_analytics.metrics import aggregate_closeout
from feedlot_analytics.relations import safe_ratio

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
result, events = load_health_analysis(root)
animals, health, scale, lots = (
    result.tables[name] for name in ("animal_closeout", "animal_health_summary", "animal_scale_stage", "lot_closeout")
)
history = animal_history(animals, health, events)
intervals = intervals_with_treatments(scale, animals, events)
overall = aggregate_closeout(animals, health).iloc[0]
outcomes = aggregate_closeout(animals, health, by=("exit_type",)).set_index("exit_type")
summary = result.reconciliation["summary"]
print(f"{summary['checks_passed']} baseline controls, {summary['checks_failed']} failed")
print(f"{len(result.tables['weather_daily'])}/704 weather days; original cattle and weather hashes verified")
display(pd.Series(summary["population"], name="observed"))
display(stage_coverage(scale, animals))

## Recorded pulls, veterinary bill and follow-up
Day on feed (DOF) is inclusive. Each animal has one first event, ordered by date and treatment ID. Early means DOF 1–30. Outcomes are source labels with undefined assessment dates. The fixed follow-up starts on the day after each index event. Death on the index date appears separately and sale within the window competes with a recorded death or repeat.

In [ ]:
history_counts = history.assign(dead=history.exit_type.eq(MORTALITY)).groupby("history").agg(
    placed=("animal_id", "size"), events=("treatment_events", "sum"), deaths=("dead", "sum"),
    direct_cents=("direct_treatment_cost_cents", "sum"), base_cad=("base_vet_cost_cad", "sum"),
    ledger_vet_cad=("vet_cost_cad", "sum"),
)
history_counts["death_pct"] = 100 * history_counts.deaths / history_counts.placed
history_counts["direct_cad"] = history_counts.direct_cents / 100
display(history_counts.drop(columns="direct_cents").round(2))
display(pd.crosstab(events.outcome.fillna("Missing"), events.exit_type, margins=True))
first = treatment_followup(history)
second = treatment_followup(history, index="second")
first["timing"] = np.where(first.first_treatment_dof.le(30), "DOF 1–30", "DOF 31+")
first["gap_to_second"] = (first.second_treatment_date - first.first_treatment_date).dt.days
display(first.first_treatment_dof.describe().round(2))
display(first.gap_to_second.describe().round(2))
for data, group in [(first, "timing"), (second.assign(index="After second event"), "index")]:
    table = data.groupby(group).agg(
        index_head=("animal_id", "size"), next_day_head=("at_risk_next_day", "sum"),
        same_day_death=("index_day_death", "sum"), deaths_1_30=("later_death_in_window", "sum"),
        deaths_through_exit=("later_death_through_exit", "sum"),
        competing_sales=("sale_in_window", "sum"), observed_to_day30=("observed_through_window", "sum"),
    )
    table["death_pct_days_1_30"] = 100 * safe_ratio(table.deaths_1_30, table.next_day_head)
    if group == "timing":
        table["repeat_days_1_30"] = data.groupby(group).repeat_in_window.sum()
        table["same_day_repeat"] = data.groupby(group).same_day_repeat.sum()
    display(table.round(3))
print(f"Direct CAD/event {overall.direct_vet_cad_per_event:,.2f}")
print(f"Direct CAD/treated head {overall.direct_treatment_cost_cents / (100 * overall.head_treated):,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.25), layout="constrained")
deaths = history.loc[history.exit_type.eq(MORTALITY)]
bins = np.arange(.5, 191.5, 7)
for axis, values, label, color in [
    (axes[0], first.first_treatment_dof, "First recorded pulls (n=2,100)", "#166a70"),
    (axes[1], deaths.financial_dof, "Deaths (n=240)", "#a35138"),
]:
    axis.hist(values, bins=bins, color=color, edgecolor="white", linewidth=.5)
    axis.set(title=label, xlabel="Inclusive day on feed", ylabel="Animals per 7-day bin")
    axis.spines[["top", "right"]].set_visible(False)
axes[0].axvline(30.5, color="#384657", linestyle="--", linewidth=1, label="Through DOF 30")
axes[0].legend(frameon=False)
plt.show()

## Timing, growth and scale availability
An interval is fully before the index event only if it ends on an earlier calendar date. It is fully after only if it starts on a later date. The before/after figures omit intervals that straddle the index. Start-day, end-day and interior events are counted separately. Later observed intervals select survivors and can still contain other treatments, particularly after first pull. The Day 35 comparison fixes health history before the weighing date and accepts later first pulls in its reference group.

In [ ]:
timing = first_pull_vs_stage(history, scale)
display(timing.groupby(["stage", "timing"]).size().unstack(fill_value=0).reindex(
    ["Arrival", "Day 35", "Day 70", "Day 105", "Closeout"]
))
event_placement = events[["treatment_id", "animal_id", "treatment_date"]].merge(
    scale.groupby("animal_id").weigh_date.agg(first_weigh="min", last_weigh="max"),
    on="animal_id", how="left", validate="many_to_one",
)
weights = scale[["animal_id", "weigh_date"]].drop_duplicates().rename(columns={"weigh_date": "treatment_date"})
event_placement = event_placement.merge(
    weights.assign(on_weigh_date=True), on=["animal_id", "treatment_date"], how="left", validate="many_to_one",
)
event_placement["coverage"] = np.select(
    [event_placement.on_weigh_date.notna(), event_placement.treatment_date.lt(event_placement.first_weigh),
     event_placement.treatment_date.gt(event_placement.last_weigh), event_placement.first_weigh.isna()],
    ["Same date as a weigh", "Before first weigh", "After last weigh", "No weigh record"], default="Between weighs",
)
display(event_placement.coverage.value_counts())
observed = intervals.assign(prior_group=intervals.prior_events.clip(upper=2))
stage_exposure = observed.groupby(["weigh_stage", "prior_group"], observed=True).agg(
    observed_head=("animal_id", "size"), interval_days=("elapsed_interval_days", "sum"),
    gain_kg=("interval_gain_kg", "sum"), inside_events=("events_inside", "sum"),
    on_start=("events_on_start", "sum"), on_end=("events_on_end", "sum"),
)
stage_exposure["elapsed_weighted_adg"] = safe_ratio(stage_exposure.gain_kg, stage_exposure.interval_days)
display(stage_exposure.round(3))

for index in ("first", "second"):
    pairs = treatment_growth_pairs(history, intervals, event=index)
    paired = pairs.dropna(subset=["adg_change"]).copy()
    pens = paired.groupby("pen_id").adg_change.agg(numerator="sum", denominator="size")
    print(index, f"paired mean change {paired.adg_change.mean():.3f}, 95% pen bootstrap {pen_rate_interval(pens)}")
    display(pairs.groupby("exit_type").agg(
        indexed=("animal_id", "size"), observed_pre=("pre_interval_adg_kg", "count"),
        observed_post=("post_interval_adg_kg", "count"), paired=("adg_change", "count"),
    ))
    display(paired.groupby(["pre_weigh_stage", "post_weigh_stage"], observed=True).agg(
        paired=("animal_id", "size"), pre_adg=("pre_interval_adg_kg", "mean"),
        post_adg=("post_interval_adg_kg", "mean"), change=("adg_change", "mean"),
        median_pre_gap=("pre_gap_days", "median"), median_post_gap=("post_gap_days", "median"),
        post_intervals_with_interior_event=("post_events_inside", lambda values: values.gt(0).sum()),
    ).round(3))
    if index == "first":
        pairs = pairs.merge(first[["animal_id", "timing"]], on="animal_id", validate="one_to_one")
        display(pairs.groupby("timing").agg(
            indexed=("animal_id", "size"), deaths=("mortality", "sum"),
            observed_pre=("pre_interval_adg_kg", "count"), observed_post=("post_interval_adg_kg", "count"),
            paired=("adg_change", "count"), mean_change=("adg_change", "mean"),
        ).round(3))

In [ ]:
landmark = growth_landmark(history, intervals)
print(f"{len(landmark):,} observed and still present after Day 35; excluded {landmark.same_day_first.sum()} same-day first pulls")
day35 = landmark.loc[~landmark.same_day_first]
display(day35.groupby("prior_treatment").agg(
    eligible=("animal_id", "size"), death=("death_through_end", "sum"), sale=("sale_through_end", "sum"),
    new_first=("new_first_in_window", "sum"), observed_adg=("next_adg", "count"),
    time_zero_adg=("landmark_adg", "mean"), next_adg=("next_adg", "mean"), change=("adg_change", "mean"),
).round(3))
landmark_results = pd.DataFrame({
    outcome: cohort_difference(day35, exposed="prior_treatment", outcome=outcome, scale=factor)
    for outcome, factor in [("death_through_end", 100), ("landmark_adg", 1), ("next_adg", 1), ("adg_change", 1)]
}).T
display(landmark_results.round(4))

## Persistent low growth
The screening cohort needs three observed Day 35, 70 and 105 intervals. The main cutoffs use all cattle weighed in the same lot, stage and date as that weighing, including ties. The earlier Phase 2B-2 sensitivity computes cutoffs among eventual complete cases and is retrospective. The Day 105 review checks strictly prior treatment. The 30-day follow-up estimates newly recorded first pulls only for animals still present and treatment-free at the landmark, excluding same-day first pulls. Death and sale are competing exits. Cohort contrasts throughout the notebook compare cattle within lot, sex, breed, frame and fixed 20 kg entry-weight bands unless specified. The reference is standardized to the eligible exposed mix on common support, with 2,000 whole-pen bootstrap draws.

In [ ]:
ranks = persistent_growth(history, intervals)
retrospective = persistent_growth(history, intervals, landmark_ranks=False)
print(f"Three observed stages {len(ranks):,}; persistent low {ranks.persistent_low.sum()}; "
      f"Phase 2B-2 complete-case definition {retrospective.persistent_low.sum()}")
display(ranks.groupby("persistent_low").agg(
    complete=("animal_id", "size"), known_prior=("prior_treatment", "sum"),
    first_same_day=("same_day_first", "sum"), lifetime_treated=("ever_treated", "sum"),
))
follow = ranks.loc[ranks.at_risk_next_day & ~ranks.prior_treatment & ~ranks.same_day_first]
display(follow.groupby("persistent_low").agg(
    at_risk=("animal_id", "size"), new_first=("new_first_in_window", "sum"),
    deaths_in_window=("death_in_window", "sum"), sales_in_window=("sale_in_window", "sum"),
))
comparison = pd.DataFrame({
    "Prior treatment, 20 kg": cohort_difference(ranks, exposed="persistent_low", outcome="prior_treatment", scale=100),
    "Prior treatment, 10 kg": cohort_difference(with_entry_band(ranks, width=10), exposed="persistent_low",
                                               outcome="prior_treatment", scale=100),
    "Prior treatment, Phase 2B-2 cutoff": cohort_difference(retrospective, exposed="persistent_low",
                                                           outcome="prior_treatment", scale=100),
    "New first pull by Day 135": cohort_difference(follow, exposed="persistent_low", outcome="new_first_in_window",
                                                  scale=100),
}).T
display(comparison.round(3))

## Mortality, timing and the financial bridge
Lifetime treatment labels below are retrospective. The daily risk sets start with entry, count exit day, end at death or competing sale, and use treatments dated strictly before that day. Comparison rates are per 1,000 animal-days on common lot and DOF support. The same-date sensitivity assigns the treatment first whenever timestamps tie. The reference rate is standardized to the exposed lot and DOF distribution. Pen resampling keeps lots nested within pens.

In [ ]:
daily = daily_risk(risk_segments(history))
same_day = daily_risk(risk_segments(history, include_event_day=True))
assert daily.head_days.sum() == overall.financial_head_days and daily.deaths.sum() == overall.head_mortality
for label, risk, limit in [
    ("Events credited next day, full follow-up", daily, None),
    ("Events credited next day, DOF 1–105", daily, 105),
    ("Same-date sensitivity, full follow-up", same_day, None),
]:
    print(label)
    supported = risk_contrasts(risk, max_dof=limit)
    display(supported.round(4))
    if limit is None:
        by_state = risk.groupby("state")[["head_days", "deaths"]].sum()
        display(by_state.assign(deaths_per_1000=1000 * by_state.deaths / by_state.head_days).round(4))
risk_period = daily.assign(dof_band=pd.cut(daily.dof, DOF_BINS, labels=DOF_LABELS))
period = risk_period.groupby("dof_band", observed=True)[["head_days", "deaths"]].sum()
display(period.assign(deaths_per_1000=1000 * period.deaths / period.head_days).round(4))
display(deaths.financial_dof.describe().round(2))
display(deaths.groupby(deaths.exit_date.dt.to_period("Q")).animal_id.size().rename("death_events_by_exit_quarter"))

display(cost_structure(overall, outcomes).round(4))
display(outcomes[[
    "head_placed", "head_treated", "direct_treatment_cost_cents", "gross_revenue_cents",
    "production_cost_cents", "all_outcome_margin_cents",
]])
print(f"Sold contribution CAD/placed {overall.sold_only_margin_cad / overall.head_placed:.2f}")
print(f"Death burden CAD/placed {overall.mortality_loss_cad / overall.head_placed:.2f}")
print(f"Net modeled CAD/placed {overall.all_outcome_margin_per_head_placed:.2f}")

In [ ]:
for key in [("breed",), ("sex",), ("frame_score",), ("origin_region",), ("placement_quarter",),
            ("origin_region", "placement_quarter")]:
    table = aggregate_closeout(animals, health, by=key)
    coverage = history.assign(dead=history.exit_type.eq(MORTALITY)).groupby(
        [*key, "pen_id"], observed=True, as_index=False
    ).dead.agg(numerator="sum", denominator="size")
    uncertainty = []
    for group, rows in coverage.groupby(list(key), observed=True):
        low, high = pen_rate_interval(rows, scale=100)
        uncertainty.append({**dict(zip(key, group, strict=True)), "pens": len(rows), "pen_low": low, "pen_high": high})
    table = table.merge(pd.DataFrame(uncertainty), on=list(key), validate="one_to_one")
    table["mortality_pct"] = 100 * table.mortality_rate
    display(table[[*key, "head_placed", "head_mortality", "mortality_pct", "pens", "pen_low", "pen_high",
                   "mortality_loss_cad"]].round(3))

lot_review = lot_health_review(lots, animals)
fields = [
    "lot_id", "placement_quarter", "origin_region", "head_placed", "head_treated", "head_mortality",
    "mortality_rate", "peer_mortality", "mortality_gap_pp", "sold_adg_kg_per_day",
    "sold_margin_contribution_per_placed", "death_burden_per_placed", "all_outcome_margin_per_head_placed",
]
print(f"Zero-death lots: {lot_review.head_mortality.eq(0).sum()}; joint review lots: {lot_review.joint_review.sum()}")
display(lot_review.nlargest(5, "mortality_gap_pp")[fields].round(4))
display(lot_review.loc[lot_review.joint_review, fields].sort_values("all_outcome_margin_per_head_placed").round(4))

## Diagnosis and protocol records
Event group tables count unique animals within each cell. An animal may contribute to several cells. Closeout and subsequent mortality use mutually exclusive first-event cohorts. Cells with fewer than 30 first events or fewer than five follow-up deaths are flagged for context. This descriptive review supports checks of recorded protocol selection, diagnosis coding and follow-up.

In [ ]:
for group in [("diagnosis",), ("drug_protocol",), ("diagnosis", "drug_protocol")]:
    event_table, cohort_table = protocol_review(events, history, health, by=group)
    event_table["direct_cad"] = event_table.direct_cost_cents / 100
    display(event_table.drop(columns="direct_cost_cents").round(2))
    review = event_table[["first_events"]].join(cohort_table)
    review["limited_support"] = review.first_events.lt(30) | review.deaths_days_1_30.lt(5)
    review["death_pct_days_1_30"] = 100 * review.death_risk_days_1_30
    display(review[[
        "first_events", "next_day_head", "day0_deaths", "deaths_days_1_30", "death_pct_days_1_30",
        "repeat_days_1_30", "head_sold", "sold_adg_kg_per_day", "sold_only_margin_per_head_sold",
        "all_outcome_margin_per_head_placed", "limited_support",
    ]].round(3))

## Carcass volume, grading and realized price
Sold observations are eligible for HCW, dressing and grade subject to the original KPI checks. Dressing divides eligible HCW by corresponding live exit kg. Realized CAD/kg divides matching gross revenue by HCW. Grade shares use all eligible graded sales. Group summaries use individual breed. Use the separate lot table to examine procurement composition and price basis. Quality labels and realized revenue have no component-level settlement detail.

In [ ]:
sold = sold_carcasses(history)
sold_total = aggregate_closeout(sold, health).iloc[0]
display(sold[["eligible_hcw_kg", "individual_yield_pct", "individual_realized_price"]].describe(
    percentiles=[.05, .25, .50, .75, .95]
).round(3))
print(f"Total HCW {sold_total.hcw_kg:,.1f} kg; live exit weight {sold_total.dressing_live_kg:,.1f} kg")
print(f"Pooled yield {100 * sold_total.dressing_yield:.4f}%; realized CAD/kg {sold_total.realized_carcass_price_cad_per_kg:.4f}")
display(outcomes.loc[[FINISHED], [
    "gross_revenue_cents", "purchase_cost_cents", "allocated_feed_cost_cents", "yardage_cost_cents",
    "vet_cost_cents", "production_cost_cents", "sold_only_margin_cents",
]] / 100)

carcass_columns = [
    "head_sold", "head_hcw_eligible", "head_dressing_eligible", "head_graded", "mean_hcw_kg", "hcw_kg",
    "dressing_yield", "prime_aaa_share", "realized_carcass_price_cad_per_kg",
    "sold_only_margin_per_head_sold", "sold_adg_kg_per_day", "sold_fcr",
]
for group in [("quality_grade",), ("breed",), ("sex",), ("frame_score",), ("treated", "repeat_treated"),
              ("origin_region",), ("placement_quarter",), ("origin_region", "placement_quarter")]:
    cohort = aggregate_closeout(sold, health, by=group)
    cohort["sale_share"] = cohort.head_sold / sold_total.head_sold
    display(cohort[[*group, *carcass_columns, "sale_share"]].round(4))
display(carcass_bands(sold, health)[["weight", "band", *carcass_columns]].round(4))
grade_ledger = aggregate_closeout(sold, health, by=("quality_grade",)).set_index("quality_grade").reindex(GRADES)
dollar_fields = ["gross_revenue_cents", "purchase_cost_cents", "allocated_feed_cost_cents",
                 "yardage_cost_cents", "vet_cost_cents", "sold_only_margin_cents"]
display((grade_ledger[dollar_fields] / 100).rename(columns=lambda name: name.replace("_cents", "_cad")).round(2))
mix_columns = ["lot_id", "placement_quarter", "origin_region", "predominant_breed", "head_sold", "mean_entry_weight_kg",
               "purchase_basis_cad_per_kg", "mean_hcw_kg", "realized_carcass_price_cad_per_kg",
               "breed_Black Angus", "breed_Charolais Cross", "breed_Hereford", "breed_Simmental Cross"]
lot_carcass_mix = lot_review[mix_columns].sort_values("mean_hcw_kg")
display(pd.concat([lot_carcass_mix.head(3), lot_carcass_mix.tail(3)]).round(3))

In [ ]:
grade_data = aggregate_closeout(sold, health, by=("quality_grade",)).set_index("quality_grade").reindex(GRADES)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), layout="constrained")
labels = [f"{label} (n={n:,})" for label, n in zip(
    ["Prime", "AAA", "AA", "Discount"], grade_data.head_graded, strict=True
)]
for axis, field, caption, limits, digits in [
    (axes[0], "mean_hcw_kg", "Mean hot carcass weight (kg)", (340, 415), 1),
    (axes[1], "realized_carcass_price_cad_per_kg", "Realized CAD per kg HCW", (7.0, 8.6), 2),
]:
    values = grade_data[field]
    axis.scatter(values, labels, color="#166a70")
    for index, value in enumerate(values):
        axis.annotate(f"{value:.{digits}f}", (value, index), xytext=(7, 0), textcoords="offset points", va="center")
    axis.set(xlabel=caption, xlim=limits)
    axis.invert_yaxis()
    axis.spines[["top", "right", "left"]].set_visible(False)
axes[1].tick_params(axis="y", labelleft=False, length=0)
plt.show()

## Comparable cohorts and management review
Continuous associations center within lot and account for individual breed, sex, frame and entry weight when applicable. All sold cattle within a lot share sex and financial DOF, so their between-lot effects are absorbed. The 95% regression intervals use pen-clustered scores. Grade revenue is examined with and without HCW as a covariate. ADG, FCR, liveweight and carcass outcomes share components, so these are descriptive relationships.

For lot triage, the review flag combines morbidity strictly above the placement quarter's 75th percentile with sold ADG and all-outcome CAD/placed below their quarter medians. Mortality exceptions use the weighted mortality rate of other lots in the same placement quarter. Thresholds screen for follow-up. The full lot review and carcass mix remain available as data frames.

In [ ]:
sold_with_cohorts = with_entry_band(sold.assign(treated=sold.treatment_events.gt(0)))
sale_contrasts = pd.DataFrame({
    outcome: cohort_difference(sold_with_cohorts, exposed="treated", outcome=outcome, scale=factor)
    for outcome, factor in [
        ("individual_financial_adg", 1), ("individual_allocated_fcr", 1), ("eligible_hcw_kg", 1),
        ("individual_yield_pct", 1), ("eligible_prime_aaa", 100), ("individual_realized_price", 1),
        ("gross_revenue_cad", 1), ("net_margin_cad", 1),
    ]
}).T
display(sale_contrasts.round(4))
patterns = [
    ("entry_weight_kg", "eligible_hcw_kg", 10, ()),
    ("exit_weight_kg", "eligible_hcw_kg", 10, ()),
    ("individual_financial_adg", "eligible_hcw_kg", .1, ()),
    ("individual_allocated_fcr", "eligible_hcw_kg", 1, ()),
    ("individual_financial_adg", "eligible_prime_aaa", .1, ()),
    ("individual_allocated_fcr", "eligible_prime_aaa", 1, ()),
    ("individual_financial_adg", "individual_yield_pct", .1, ()),
    ("eligible_hcw_kg", "gross_revenue_cad", 10, ()),
    ("eligible_prime_aaa", "gross_revenue_cad", 1, ()),
    ("eligible_prime_aaa", "gross_revenue_cad", 1, ("eligible_hcw_kg",)),
]
fits = []
for predictor, outcome, increment, extra in patterns:
    controls = tuple(field for field in ("entry_weight_kg", *extra) if field != predictor)
    fits.append({
        "predictor": predictor, "outcome": outcome, "increment": increment, "controls": ", ".join(controls),
        **within_lot_slope(sold, predictor=predictor, outcome=outcome, increment=increment, numeric=controls,
                           categorical=("breed", "sex", "frame_score")),
    })
display(pd.DataFrame(fits).round(4))

## Interpretation and reproducibility
Cluster intervals describe sensitivity within the simulated set. They do not establish real-world risk or causal effects. Observe eligibility at each time zero, retain missing interval counts and account for exits. Treatment dates have no times of day, outcome labels have no assessment dates, and severity, treatment assignment, dose, independent veterinary follow-up and necropsies are unavailable. The sale cohort is conditional on survival. Allocated feed is a bookkeeping estimate and independent settlement adjustments are unavailable.

Run from the repository root with Python 3.11+, install the notebook extra, and execute as shown in the findings report. The canonical build validates the original source and weather hashes, all 661 controls and 704 weather days before analysis.